# Fire-safety YOLO26-seg training on Kaggle

**Before running**
1. *Add Input* → your private dataset with the exported split (the folder that contains `data.yaml`, `train/`, `val/`, `test/`).
2. *Settings* → **Accelerator: GPU T4** (or P100) and **Internet: On** (needed for `pip` and the pretrained weights).
3. Edit the settings cell below (run name, model, image size, batch).

**Fine-tuning an earlier run:** *Add Input* → *Notebook Output Files* → the version that trained it, then set
`FINETUNE_FROM` to its run name (and `OWN_DATA_ONLY = True` to train on own images only).

**Final model (D-25):** set `TRAIN_ON_TEST = True`, `SEED = 0` and keep every other setting as in the seed runs.
The test images (held-out schools) are added to training, validation stays the same, and the run is not evaluated on test.

**Run it in the background:** *Save Version* → *Save & Run All (Commit)*. You can close the browser; when it finishes,
open the version → *Output* → download `<RUN_NAME>.zip`.

Keep each run under Kaggle's per-session time limit (about 12 h). Check the time of the first epochs in the log.


In [ ]:
# ---------------- SETTINGS (edit per experiment) ----------------
RUN_NAME = "FINAL_y26s-seg_own+public_1024_train+heldout"
MODEL    = "yolo26s-seg.pt"      # yolo26n-seg.pt / yolo26s-seg.pt / yolo26m-seg.pt
IMGSZ    = 1024
BATCH    = 16                    # 8 or 16: same learning (gradients accumulate to 64), 16 is a bit faster
CACHE    = "ram"                 # "ram" = images decoded once and kept in memory (faster); "none" = off
EPOCHS   = 100
PATIENCE = 30
SEED     = 0
NOTES    = "Final model (D-25): small, own + public, seed 0, trained on train + held-out schools, v2 labels"

# Fine-tuning (leave FINETUNE_FROM empty for a normal run)
FINETUNE_FROM = ""               # run name of an earlier run, e.g. "EXP004_y26n-seg_own+public_1024";
                                 # attach that notebook version's output as an input. Its best.pt replaces MODEL.
OWN_DATA_ONLY = False            # True: use only own images of the attached split (drops fireart_/firenet_ files)
PUBLIC_PREFIXES = ("fireart_", "firenet_")

# Final model (D-25): also train on the test images (held-out schools). Validation is unchanged
# (early stopping and best.pt as before); there is no test evaluation, because the test images
# become training data. Keep all other settings as in the seed runs, SEED = 0.
TRAIN_ON_TEST = True
ULTRALYTICS_VERSION = "8.4.171"  # same version as on the laptop, for comparable results
# -----------------------------------------------------------------
import os
KAGGLE_IN  = os.environ.get("KAGGLE_IN", "/kaggle/input")
KAGGLE_OUT = os.environ.get("KAGGLE_OUT", "/kaggle/working")


In [ ]:
import subprocess, sys
if not os.environ.get("SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU - set Settings > Accelerator to GPU before running a real experiment")


In [ ]:
# Find the uploaded data.yaml and write a copy whose 'path' points to Kaggle's input folder
import glob, yaml
cands = [p for p in glob.glob(os.path.join(KAGGLE_IN, "**", "data.yaml"), recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "train", "images"))]
if len(cands) != 1:
    raise SystemExit(f"Expected exactly one exported split with data.yaml, found: {cands}")
DATA_ROOT = os.path.dirname(cands[0])
cfg = yaml.safe_load(open(cands[0], encoding="utf-8"))
cfg["path"] = DATA_ROOT
for k in ("train", "val", "test"):
    cfg[k] = f"{k}/images"
os.makedirs(KAGGLE_OUT, exist_ok=True)
DATA_YAML = os.path.join(KAGGLE_OUT, "data.yaml")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
used = {}
for k in ("train", "val", "test"):
    files = sorted(f for f in os.listdir(os.path.join(DATA_ROOT, k, "images")) if f.lower().endswith(IMG_EXT))
    if OWN_DATA_ONLY:
        files_used = [f for f in files if not f.lower().startswith(PUBLIC_PREFIXES)]
        print(f"{k:5} {len(files_used):6d} own images (of {len(files)})")
    else:
        files_used = files
        print(f"{k:5} {len(files_used):6d} images")
    used[k] = [os.path.join(DATA_ROOT, k, "images", f) for f in files_used]
    if OWN_DATA_ONLY:
        lst = os.path.join(KAGGLE_OUT, f"{k}_own.txt")
        with open(lst, "w", encoding="utf-8") as fh:
            fh.write("\n".join(used[k]) + "\n")
        cfg[k] = lst

if TRAIN_ON_TEST:
    # training list = train + test images; validation unchanged; no test split in the yaml
    lst = os.path.join(KAGGLE_OUT, "train_plus_test.txt")
    with open(lst, "w", encoding="utf-8") as fh:
        fh.write("\n".join(used["train"] + used["test"]) + "\n")
    cfg["train"] = lst
    cfg.pop("test", None)
    print(f"TRAIN_ON_TEST: training on {len(used['train'])} + {len(used['test'])} = "
          f"{len(used['train']) + len(used['test'])} images; validation {len(used['val'])} images; "
          "no test evaluation")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print("data root:", DATA_ROOT, "| classes:", len(cfg["names"]))

# Start from an earlier run's best.pt
if FINETUNE_FROM:
    w = [p for p in glob.glob(os.path.join(KAGGLE_IN, "**", "weights", "best.pt"), recursive=True)
         if os.path.basename(os.path.dirname(os.path.dirname(p))) == FINETUNE_FROM]
    if len(w) != 1:
        raise SystemExit(f"Expected one best.pt for run {FINETUNE_FROM!r} in the inputs, found: {w}. "
                         "Attach that notebook version's output as an input.")
    MODEL = w[0]
    print("fine-tuning from:", MODEL)


In [ ]:
TRAIN_EVAL = r'''#!/usr/bin/env python3
"""
train_eval.py - train a YOLO26 segmentation model, then evaluate best.pt on the TEST split
and save the results needed for the paper.

Uses the data.yaml written by make_split.py (either <root>/data_split.yaml or
<export>/data.yaml). The test set is only touched once, after training.

Example (own data only, nano model):
  python train_eval.py --data "D:/.../FSE-360_v1_split/data.yaml" --model yolo26n-seg.pt ^
                       --name EXP001_y26n-seg_own-only

Final model (D-25): --skip-test trains only. Use it when the training list also contains the
test images; the run is then logged as "not evaluated on test".

Outputs (in <project>/<name>/):
  weights/best.pt, results.csv, args.yaml, plots      (Ultralytics training output)
  test/                                               (test-set plots, confusion matrix)
  test_metrics_per_class.csv                          box + mask P, R, mAP50, mAP50-95
  test_metrics_overall.csv                            one row, also appended to the log
<project>/experiment_log.csv                          one row per run - your results table
"""
import argparse
import csv
import time
from datetime import datetime
from pathlib import Path

from ultralytics import YOLO


def get_args():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--data", required=True, help="data.yaml from make_split.py")
    p.add_argument("--model", default="yolo26n-seg.pt")
    p.add_argument("--name", required=True, help="run name, e.g. EXP001_y26n-seg_own-only")
    p.add_argument("--project", default="experiments")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=16, help="lower to 8 or 4 on out-of-memory")
    p.add_argument("--patience", type=int, default=30, help="stop early if val stops improving")
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--device", default="0", help="0 = first GPU, cpu = CPU")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--cache", default="none", choices=["none", "ram", "disk"],
                   help="cache decoded images: ram = fastest (needs free RAM), "
                        "disk = .npy files next to the images (needs a writable dataset folder)")
    p.add_argument("--notes", default="", help="free text for the experiment log")
    p.add_argument("--eval-only", action="store_true",
                   help="skip training, evaluate <project>/<name>/weights/best.pt on test")
    p.add_argument("--skip-test", action="store_true",
                   help="train only, no test evaluation (final model whose training data "
                        "includes the test images, D-25)")
    a = p.parse_args()
    if a.skip_test and a.eval_only:
        p.error("--skip-test and --eval-only cannot be combined")
    return a


def f4(x):
    return f"{float(x):.4f}"


def main():
    a = get_args()
    run_dir = Path(a.project).resolve() / a.name
    t0 = time.time()

    # ---------- train ----------
    if not a.eval_only:
        if (run_dir / "weights").exists():
            raise SystemExit(f"Run folder already exists: {run_dir}\n"
                             f"Use a new --name, or --eval-only to re-evaluate that run.")
        model = YOLO(a.model)
        model.train(data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch,
                    patience=a.patience, workers=a.workers, device=a.device, seed=a.seed,
                    deterministic=True, project=str(run_dir.parent), name=run_dir.name,
                    exist_ok=False, plots=True,
                    cache=False if a.cache == "none" else a.cache)
        run_dir = Path(model.trainer.save_dir).resolve()  # the folder Ultralytics really used
    train_min = (time.time() - t0) / 60

    # ---------- final model: test images are training data, so no test evaluation ----------
    if a.skip_test:
        row = {"time": datetime.now().strftime("%Y-%m-%d %H:%M"), "run": run_dir.name,
               "model": Path(a.model).name, "data": a.data, "epochs": a.epochs,
               "imgsz": a.imgsz, "batch": a.batch, "seed": a.seed, "cache": a.cache,
               "box_mAP50": "", "box_mAP50-95": "", "mask_mAP50": "", "mask_mAP50-95": "",
               "box_P": "", "box_R": "", "mask_P": "", "mask_R": "",
               "ms_per_img_inference": "", "train_minutes": f"{train_min:.1f}",
               "notes": ("NOT EVALUATED ON TEST: test images are in the training data. " + a.notes).strip()}
        with open(run_dir / "test_metrics_overall.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            w.writeheader()
            w.writerow(row)
        log = run_dir.parent / "experiment_log.csv"
        new = not log.exists()
        with open(log, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"\nTraining done ({train_min:.1f} min). No test evaluation: the test images are "
              f"part of the training data.\nWeights: {run_dir / 'weights' / 'best.pt'}\nLog:     {log}")
        return

    # ---------- evaluate best.pt on the test split ----------
    best = run_dir / "weights" / "best.pt"
    if not best.exists():
        raise SystemExit(f"best.pt not found: {best}")
    m = YOLO(str(best)).val(data=a.data, split="test", imgsz=a.imgsz, batch=a.batch,
                            device=a.device, project=str(run_dir), name="test",
                            exist_ok=True, plots=True)
    names = m.names

    # per class (only classes that have test instances appear in ap_class_index)
    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        c = int(c)
        rows.append([c, names[c],
                     f4(m.box.p[i]), f4(m.box.r[i]), f4(m.box.ap50[i]), f4(m.box.ap[i]),
                     f4(m.seg.p[i]), f4(m.seg.r[i]), f4(m.seg.ap50[i]), f4(m.seg.ap[i])])
    present = {r[0] for r in rows}
    for c in sorted(names):
        if c not in present:
            rows.append([c, names[c]] + ["no test instances"] + [""] * 7)
    rows.sort(key=lambda r: r[0])
    head = ["id", "class", "box_P", "box_R", "box_mAP50", "box_mAP50-95",
            "mask_P", "mask_R", "mask_mAP50", "mask_mAP50-95"]
    with open(run_dir / "test_metrics_per_class.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(head)
        w.writerows(rows)

    # overall + log
    overall = {
        "time": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "run": run_dir.name,
        "model": "best.pt (eval-only)" if a.eval_only else (
            f"{Path(a.model).parent.parent.name}/best.pt (fine-tune)"
            if Path(a.model).name in ("best.pt", "last.pt") else Path(a.model).name),
        "data": a.data,
        "epochs": a.epochs, "imgsz": a.imgsz, "batch": a.batch, "seed": a.seed,
        "cache": a.cache,
        "box_mAP50": f4(m.box.map50), "box_mAP50-95": f4(m.box.map),
        "mask_mAP50": f4(m.seg.map50), "mask_mAP50-95": f4(m.seg.map),
        "box_P": f4(m.box.mp), "box_R": f4(m.box.mr),
        "mask_P": f4(m.seg.mp), "mask_R": f4(m.seg.mr),
        "ms_per_img_inference": f"{m.speed.get('inference', 0):.1f}",
        "train_minutes": "" if a.eval_only else f"{train_min:.1f}",
        "notes": a.notes,
    }
    with open(run_dir / "test_metrics_overall.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(overall))
        w.writeheader()
        w.writerow(overall)
    log = run_dir.parent / "experiment_log.csv"
    new = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(overall))
        if new:
            w.writeheader()
        w.writerow(overall)

    print("\n=== TEST SET (held-out locations) ===")
    print(f"mask mAP50-95 {overall['mask_mAP50-95']}  mask mAP50 {overall['mask_mAP50']}  "
          f"box mAP50-95 {overall['box_mAP50-95']}")
    print(f"{'class':<24}{'mask mAP50':>11}{'mask mAP50-95':>15}")
    for r in rows:
        if r[2] == "no test instances":
            print(f"{r[1]:<24}{'-':>11}{'-':>15}   (no test instances)")
        else:
            print(f"{r[1]:<24}{r[8]:>11}{r[9]:>15}")
    print(f"\nSaved: {run_dir / 'test_metrics_per_class.csv'}\nLog:   {log}")


if __name__ == "__main__":  # required on Windows (data-loader workers)
    main()
'''
os.chdir(KAGGLE_OUT)
with open('train_eval.py', 'w', encoding='utf-8') as f:
    f.write(TRAIN_EVAL)
print('train_eval.py written to', KAGGLE_OUT)


In [ ]:
# Train + evaluate best.pt on the test split (output is streamed below)
cmd = [sys.executable, "train_eval.py", "--data", DATA_YAML, "--model", MODEL,
       "--name", RUN_NAME, "--project", os.path.join(KAGGLE_OUT, "experiments"),
       "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--epochs", str(EPOCHS),
       "--patience", str(PATIENCE), "--seed", str(SEED), "--workers", "4", "--cache", CACHE,
       "--device", DEVICE, "--notes", NOTES]
if TRAIN_ON_TEST:
    cmd.append("--skip-test")
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=KAGGLE_OUT)
if r.returncode != 0:
    raise SystemExit(f"train_eval.py failed with exit code {r.returncode}")


In [ ]:
# Pack everything from this run into one zip for download (Output tab)
import shutil
run_dir = os.path.join(KAGGLE_OUT, "experiments", RUN_NAME)
if not os.path.isdir(run_dir):
    raise SystemExit(f"Run folder not found: {run_dir}")
shutil.copy(os.path.join(KAGGLE_OUT, "experiments", "experiment_log.csv"),
            os.path.join(run_dir, "experiment_log_row.csv"))
shutil.copy(DATA_YAML, os.path.join(run_dir, "data_kaggle.yaml"))
zip_path = shutil.make_archive(os.path.join(KAGGLE_OUT, RUN_NAME), "zip", run_dir)
print("Download:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
